# 03 - Dropout ablation (dropout ON vs dropout OFF)

**What this notebook does**: trains the same MiniConvNet twice on the `faithful` split - once with
`dropout_rate = 0.5`, once with `dropout_rate = 0.0` - for the **full epoch budget**, and writes a
clean 2-row table to `outputs/ablation_dropout.csv`.

**What must already exist**: split CSVs from notebook 00. Run notebook 02 first if you want the
non-ablation baselines for context.

**Design decisions baked in**
* Full epoch budget for both arms (`EPOCHS_ABLATION`) - a truncated run makes dropout look worse
  than it is, because regularised models converge more slowly.
* Both arms are collapse-checked (LESSON 3). A collapsed arm invalidates the comparison; it must be
  re-run, not reported.
* Everything else (seed, learning rate, clipnorm, callbacks, split) is held identical, so the only
  difference between the two rows is dropout.
* The ablation runs on the `flatten` head by default because that is the paper-sized variant; the
  `ARCH_FOR_ABLATION` switch below lets you repeat it for `gap`.

**What "looks right"**: two rows in the ablation table, both `status = ok`. A modest difference in
test accuracy either way is a normal outcome on a dataset this small - what would *not* be normal is
one arm sitting at ~0.25 accuracy (chance) or being flagged as collapsed.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('ablation epoch budget:', EPOCHS_ABLATION)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_miniconvnet, count_params
from src.train_utils import (set_global_seeds, gpu_report, compile_model, optimizer_summary,
                            class_weights_for, make_callbacks, save_history, plot_history,
                            final_epoch_summary, run_name_for)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                               plot_confusion_matrix, record_ablation, load_results)

set_global_seeds(SEED)
print(gpu_report())

In [ ]:
# Ablation configuration - change these two lines to repeat the ablation elsewhere.
ARCH_FOR_ABLATION = 'flatten'   # 'flatten' (paper-sized) or 'gap'
SPLIT_FOR_ABLATION = 'faithful' # keep 'faithful' so the comparison is paper-comparable

print('arch :', ARCH_FOR_ABLATION)
print('split:', SPLIT_FOR_ABLATION)

## 1. Data

**Looks right**: the same counts you saw in notebook 00 for this split.

In [ ]:
sdf = load_split(SPLIT_FOR_ABLATION)
train_ds, val_ds, test_ds, frames = make_split_datasets(sdf)
class_weight = class_weights_for(SPLIT_FOR_ABLATION, frames['train']['label'].values)

print(split_counts(sdf))
print('class_weight:', class_weight if class_weight else 'None')

## 2. Ablation arm runner

One function, used twice with the only difference being `dropout_rate`. The seed is re-set inside it
so both arms start from the same initialisation stream.

In [ ]:
def run_ablation_arm(dropout_rate, tag):
    set_global_seeds(SEED)
    run_name = run_name_for('ablation', ARCH_FOR_ABLATION, SPLIT_FOR_ABLATION, tag)

    model = build_miniconvnet(ARCH_FOR_ABLATION, dropout_rate=dropout_rate)
    compile_model(model)
    n_dropout = sum(1 for l in model.layers if l.__class__.__name__ == 'Dropout')
    print('run         :', run_name)
    print('dropout_rate:', dropout_rate, '| Dropout layers in model:', n_dropout)
    print('params      :', count_params(model))
    print('optimizer   :', optimizer_summary(model))

    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_ABLATION,
                        class_weight=class_weight,
                        callbacks=make_callbacks(run_name), verbose=2)
    save_history(history, run_name)
    print('\n', final_epoch_summary(history))

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    print('\ntest metrics:', {k: round(v, 4) for k, v in metrics.items()})
    print()
    print_collapse_report(collapse, run_name)

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)

    record_ablation({
        'run_name': run_name,
        'arch_variant': ARCH_FOR_ABLATION,
        'split_variant': SPLIT_FOR_ABLATION,
        'dropout_enabled': bool(dropout_rate and dropout_rate > 0),
        'dropout_rate': dropout_rate,
        'accuracy': round(metrics['accuracy'], 6),
        'f1_macro': round(metrics['f1_macro'], 6),
        'cohen_kappa': round(metrics['cohen_kappa'], 6),
        'mcc': round(metrics['mcc'], 6),
        'epochs_trained': final_epoch_summary(history)['epochs_trained'],
        'status': collapse['status'],
        'notes': f'full budget {EPOCHS_ABLATION} epochs, identical seed/lr/clipnorm across arms',
    })
    return {'run_name': run_name, 'metrics': metrics, 'collapse': collapse,
            'history': history, 'y_true': y_true, 'y_pred': y_pred}

## 3. Arm 1 - dropout ON (rate 0.5)

**Looks right**: `Dropout layers in model: 1`, and a train/val accuracy gap that is *smaller* than
the dropout-off arm.

In [ ]:
arm_on = run_ablation_arm(DROPOUT_RATE, 'dropout_on')

## 4. Arm 2 - dropout OFF (rate 0.0)

**Looks right**: `Dropout layers in model: 0` - if this prints 1, the rate was not threaded through
and the ablation is meaningless.

In [ ]:
arm_off = run_ablation_arm(0.0, 'dropout_off')

## 5. Comparison

**Looks right**: exactly 2 rows in `ablation_dropout.csv`, both `ok`. Report the delta together with
the train/val gap - dropout's effect on *overfitting* is often clearer than its effect on test
accuracy at this dataset size.

In [ ]:
def gap_of(res):
    s = final_epoch_summary(res['history'])
    return round(s.get('final_accuracy', float('nan')) - s.get('final_val_accuracy', float('nan')), 4)

compare = pd.DataFrame([
    {'arm': 'dropout ON (0.5)', **{k: round(v, 4) for k, v in arm_on['metrics'].items()},
     'train_minus_val_acc': gap_of(arm_on), 'status': arm_on['collapse']['status']},
    {'arm': 'dropout OFF (0.0)', **{k: round(v, 4) for k, v in arm_off['metrics'].items()},
     'train_minus_val_acc': gap_of(arm_off), 'status': arm_off['collapse']['status']},
])
print(compare[['arm', 'accuracy', 'f1_macro', 'cohen_kappa', 'mcc',
               'train_minus_val_acc', 'status']].to_string(index=False))

delta = arm_on['metrics']['accuracy'] - arm_off['metrics']['accuracy']
print(f'\ntest accuracy delta (ON - OFF): {delta:+.4f}')
if arm_on['collapse']['collapsed'] or arm_off['collapse']['collapsed']:
    print('!!! at least one arm collapsed - the comparison is INVALID, re-run before reporting.')

In [ ]:
abl = load_results('ablation')
print('outputs/ablation_dropout.csv')
print(abl.to_string(index=False))
print('\nnext: 04_cross_validation.ipynb')